In [1]:
# Importation des modules
# Import bibliothèque de manipulation de dataframe
import pandas as pd

# Import des bibliothèques de viz
import matplotlib.pyplot as plt
import seaborn as sns

# Import split data
from sklearn.model_selection import train_test_split

# Import modèles de ML Supervisé Régression
from sklearn.linear_model import LinearRegression

# Import modèles de ML Supervisé Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Import modèle de ML NON Supervisé
from sklearn.neighbors import NearestNeighbors

# Import des métriques
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import outil standardisation de la donnée
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer

# Import pipeline
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

# Gestion des warnings
import warnings

import ast

In [2]:
# Custom transformer for MultiLabelBinarizer
class MultiLabelBinarizerPipelineFriendly(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

In [3]:
# Récuperation du df
df = pd.read_csv('../ressources/df_v4.csv', sep=';', encoding='utf-8')
df.isna().sum()

frenchTitle      0
genres           0
averageRating    0
numVotes         0
actor1           0
actor2           0
actor3           1
directors        0
decade           0
dtype: int64

In [4]:
# On remplace les valeurs manquantes par la valeur 'Unknwown'
df.dropna(subset=['actor2', 'actor3'], inplace=True)
print(df.isna().sum())
df.fillna('Unknown', inplace=True)

frenchTitle      0
genres           0
averageRating    0
numVotes         0
actor1           0
actor2           0
actor3           0
directors        0
decade           0
dtype: int64


In [5]:
# On concatene les acteurs pour en faire une colonne unique pour la passer dans un MultiLabelBinarizer
df['actors'] = df['actor1'].map(lambda x: [x]) + df['actor2'].map(lambda x: [x]) + df['actor3'].map(lambda x: [x])

# On supprime les colonnes inutiles
df = df.drop(columns=['actor1', 'actor2', 'actor3'])

# on enleve les [] et les '' de la colonne actors
df['actors'] = df['actors'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df

,frenchTitle,genres,averageRating,numVotes,directors,decade,actors
0,Intolérance,"History, Drama",7.7,17434,D.W. Griffith,1910,"[Lillian Gish, Robert Harron, Mae Marsh]"
1,Le lys brisé,"Drama, Romance",7.2,11483,D.W. Griffith,1910,"[Lillian Gish, Richard Barthelmess, Donald Crisp]"
2,J'accuse,"History, War, Drama, Horror, Romance",7.7,2249,Abel Gance,1910,"[Romuald Joubé, Maxime Desjardins, Séverin-Mars]"
3,L'admirable Crichton,"Drama, Adventure, Fantasy",7.0,2032,Cecil B. DeMille,1910,"[Thomas Meighan, Theodore Roberts, Raymond Hat..."
4,Le signe de Zorro,"Adventure, Drama, Western, Action, Romance",7.0,2946,Fred Niblo,1920,"[Douglas Fairbanks, Marguerite De La Motte, No..."
...,...,...,...,...,...,...,...
17266,Herself,Drama,7.0,5127,Phyllida Lloyd,2020,"[Molly McCann, Clare Dunne, Ruby Rose O'Hara]"
17267,Enemy Lines,"Drama, Action, War",4.6,2045,Anders Banke,2020,"[Ed Westwick, John Hannah, Tom Wisdom]"
17268,Eight for Silver,"Mystery, Fantasy, Horror",6.2,20425,Sean Ellis,2020,"[Boyd Holbrook, Kelly Reilly, Alistair Petrie]"
17269,Le lion,Comedy,5.5,1522,Ludovic Colbeau-Justin,2020,"[Dany Boon, Philippe Katerine, Anne Serra]"


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Preprocessor
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
X = df.drop(columns=['frenchTitle'])
films_non_standardise = X.iloc[:3]

In [7]:
# Fonction pour alourdir la valeur des colonnes
def multiply_block(X, factor):
    return X * factor

In [12]:
# Preprocessor pour standardiser les colonnes numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('actors', Pipeline([
            ('mlb', MultiLabelBinarizerPipelineFriendly()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 2))),
            ]), 'actors'),
        ('directors', Pipeline([
            ('mlb', MultiLabelBinarizerPipelineFriendly()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.8))),
            ]), 'directors'),
        ('genres', MultiLabelBinarizerPipelineFriendly(), 'genres'),
        ('decade', Pipeline([
            ('encoder', OrdinalEncoder()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.5))),
            ]), ['decade']),
        ('scaler', StandardScaler(), ['averageRating']),
        ('numVotes', Pipeline([
            ('scaler', StandardScaler()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.2))),
            ]), ['numVotes'])
    ]
)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Pipeline
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
# Création du pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', NearestNeighbors(n_neighbors=11))
    ]
)

pipeline.fit(X)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('actors',
                                                  Pipeline(steps=[('mlb',
                                                                   MultiLabelBinarizerPipelineFriendly()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000023CC59B58A0>))]),
                                                  'actors'),
                                                 ('directors',
                                                  Pipeline(steps=[('mlb',
                                                                   MultiLabelBinarizerPipelineFriendly()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda...
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000023CC5789EE0>))]),
                                                  ['decade']),
                                                 ('scaler', StandardScaler(),
                                                  ['averageRating']),
                                                 ('numVotes',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000023CC5789B20>))]),
                                                  ['numVotes'])])),
                ('knn', NearestNeighbors(n_neighbors=11))])

In [14]:
point_test = X.iloc[:3]
# Prédiction des voisins les plus proches
X_test_transformed = pipeline.named_steps['preprocessor'].transform(point_test)

distances, indices = pipeline.named_steps['knn'].kneighbors(X_test_transformed)
# Affichage des indices des voisins les plus proches
print("Indices des voisins les plus proches :", indices)
# Affichage des distances des voisins les plus proches
print("Distances des voisins les plus proches :", distances)


Indices des voisins les plus proches : [[   0    9    6    1  117   39   36  240  263  257   73]
 [   1    6    9   36    0  582  154  327   10   49  296]
 [   2  742  431  255   21 1134   59  413  318  721  248]]
Distances des voisins les plus proches : [[0.         4.51388846 4.93712158 5.01949027 5.7194126  5.76352596
  5.81082272 5.85194791 5.8652558  5.86906025 5.93973522]
 [0.         3.35527349 4.61062574 4.95683066 5.01949027 5.02970699
  5.06832816 5.6036701  5.61942789 5.65030814 5.66315002]
 [0.         5.61006212 5.61829476 5.70306411 5.72409114 5.73150027
  5.76552227 5.80357387 5.80540014 5.81166062 5.81420891]]


In [15]:
titres = ['Spider-Man', "Intouchables", 'Avatar']  # ou d’autres

for titre in titres:
    film_cible = df[df['frenchTitle'].str.contains(titre, case=False, na=False)]
    if film_cible.empty:
        print(f"Film '{titre}' non trouvé.")
        continue

    idx_film = film_cible.index[0]
    film_non_standardise = df.drop(columns=['frenchTitle']).loc[[idx_film]]
    film_transforme = pipeline.named_steps['preprocessor'].transform(film_non_standardise)
    distances, indices = pipeline.named_steps['knn'].kneighbors(film_transforme)

    print(f"\n🎬 Film : {df.loc[idx_film, 'frenchTitle']} (Index: {idx_film})")
    print(f"  Note moyenne : {df.loc[idx_film, 'averageRating']}")
    neighbor_original_indices = X.iloc[indices[0]].index
    neighbor_info = df.loc[neighbor_original_indices][['frenchTitle', 'averageRating', 'numVotes', 'decade', 'genres', 'actors', 'directors']]
    print("  Voisins :")
    display(neighbor_info)


🎬 Film : Spider-Man (Index: 4820)
  Note moyenne : 7.4
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
4820,Spider-Man,7.4,925217,2000,"Fantasy, Adventure, Action, Sci-Fi","[Tobey Maguire, Kirsten Dunst, Willem Dafoe]",Sam Raimi
6162,Spider-Man 2,7.5,748381,2000,"Fantasy, Adventure, Action, Sci-Fi","[Tobey Maguire, Kirsten Dunst, Alfred Molina]",Sam Raimi
7086,Spider-Man 3,6.3,670219,2000,"Fantasy, Adventure, Action, Sci-Fi","[Tobey Maguire, Kirsten Dunst, Topher Grace]",Sam Raimi
10932,Aquaman,6.8,538556,2010,"Adventure, Action, Fantasy","[Jason Momoa, Amber Heard, Willem Dafoe]",James Wan
12518,La Grande Muraille,5.9,153775,2010,"Adventure, Action, Fantasy","[Matt Damon, Tian Jing, Willem Dafoe]",Yimou Zhang
17146,Doctor Strange in the Multiverse of Madness,6.9,516821,2020,"Adventure, Action, Fantasy","[Benedict Cumberbatch, Elizabeth Olsen, Chiwet...",Sam Raimi
6975,John Carter,6.6,293334,2010,"ScienceFiction, Adventure, Action, Sci-Fi","[Taylor Kitsch, Lynn Collins, Willem Dafoe]",Andrew Stanton
4632,Small Soldiers,6.3,108743,1990,"Adventure, Action, Fantasy, Comedy, ScienceFic...","[Kirsten Dunst, Gregory Smith, David Cross]",Joe Dante
13496,Midnight Special,6.6,85593,2010,"Adventure, Mystery, Sci-Fi, Drama, ScienceFiction","[Michael Shannon, Joel Edgerton, Kirsten Dunst]",Jeff Nichols
10570,Hunger Games,7.2,1045002,2010,"Adventure, Sci-Fi, Action, Fantasy, ScienceFic...","[Jennifer Lawrence, Josh Hutcherson, Liam Hems...",Gary Ross



🎬 Film : Intouchables (Index: 11649)
  Note moyenne : 8.5
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
11649,Intouchables,8.5,980472,2010,"Drama, Comedy","[François Cluzet, Omar Sy, Anne Le Ny]","Olivier Nakache, Éric Toledano"
14138,Samba,6.7,17664,2010,"Drama, Comedy, Romance","[Omar Sy, Charlotte Gainsbourg, Tahar Rahim]","Olivier Nakache, Éric Toledano"
7799,Nos jours heureux,6.8,5120,2000,"Comedy, Family","[Jean-Paul Rouve, Marilou Berry, Omar Sy]","Olivier Nakache, Éric Toledano"
16923,Hors normes,7.4,11968,2010,"Drama, Comedy","[Vincent Cassel, Reda Kateb, Hélène Vincent]","Olivier Nakache, Éric Toledano"
15999,L'école buissonnière,6.9,2186,2010,"Drama, Comedy, Family","[François Cluzet, Jean Scandel, Eric Elmosnino]",Nicolas Vanier
10628,Tellement proches,6.4,1788,2000,"Drama, Comedy","[Vincent Elbaz, Isabelle Carré, François-Xavie...","Olivier Nakache, Éric Toledano"
10782,Les Petits Mouchoirs,7.1,27734,2010,"Drama, Comedy","[François Cluzet, Marion Cotillard, Benoît Mag...",Guillaume Canet
15276,Demain tout commence,7.3,29524,2010,"Drama, Comedy","[Omar Sy, Clémence Poésy, Antoine Bertrand]",Hugo Gélin
12727,Un métier sérieux,6.6,1940,2020,"Drama, Comedy","[Vincent Lacoste, François Cluzet, Louise Bour...",Thomas Lilti
15278,Médecin de campagne,6.5,3424,2010,"Drama, Comedy","[François Cluzet, Marianne Denicourt, Christop...",Thomas Lilti



🎬 Film : Avatar (Index: 7976)
  Note moyenne : 7.9
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
7976,Avatar,7.9,1432784,2000,"ScienceFiction, Adventure, Action, Fantasy","[Sam Worthington, Zoe Saldaña, Sigourney Weaver]",James Cameron
11488,Avatar : La Voie de l'eau,7.5,535737,2020,"ScienceFiction, Adventure, Action, Fantasy","[Sam Worthington, Zoe Saldaña, Sigourney Weaver]",James Cameron
14534,Les Gardiens de la Galaxie Vol. 2,7.6,796731,2010,"ScienceFiction, Adventure, Action, Comedy","[Chris Pratt, Zoe Saldaña, Dave Bautista]",James Gunn
2195,"Aliens, le retour",8.4,810267,1980,"Adventure, ScienceFiction, Action, Horror, Thr...","[Sigourney Weaver, Michael Biehn, Carrie Henn]",James Cameron
10642,Star Trek Into Darkness,7.7,504320,2010,"ScienceFiction, Adventure, Action, Sci-Fi","[Chris Pine, Zachary Quinto, Zoe Saldaña]",J.J. Abrams
11543,La colère des Titans,5.7,199631,2010,"Adventure, Action, Fantasy","[Sam Worthington, Liam Neeson, Rosamund Pike]",Jonathan Liebesman
5075,Galaxy Quest,7.4,185985,1990,"Adventure, ScienceFiction, Comedy, Sci-Fi","[Tim Allen, Sigourney Weaver, Alan Rickman]",Dean Parisot
8151,Le Choc des Titans,5.8,299476,2010,"Adventure, Action, Fantasy","[Sam Worthington, Liam Neeson, Ralph Fiennes]",Louis Leterrier
9180,Spider-Man: No Way Home,8.2,954809,2020,"ScienceFiction, Adventure, Action, Fantasy","[Tom Holland, Zendaya, Benedict Cumberbatch]",Jon Watts
8471,Le Hobbit : Un Voyage Inattendu,7.8,903967,2010,"Adventure, Action, Fantasy","[Martin Freeman, Ian McKellen, Richard Armitage]",Peter Jackson
